# Session 7 · Make the cost visible, then govern it

Queen City Trip Analytics, a fictional South End, Charlotte company, prices its dashboards and governs what its first fleet customer can see.

In [ ]:
source ~/dsba6190-live-demo-07/env.sh && echo "$DS   $BUCKET   $ANALYST" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The estimator prices a query before it runs

In [ ]:
bq --project_id="$PROJECT" --location=US --quiet query --use_legacy_sql=false --dry_run \
  "SELECT pickup_datetime, passenger_count, fare_amount FROM \`$TAXI\`"

In [ ]:
bq --project_id="$PROJECT" show --format=json "${TAXI/./:}" | jq -M '{numRows, numBytes}'

## Step 2 · `SELECT *`

In [ ]:
dry "SELECT * FROM \`$TAXI\`"

## Step 3 · Three named columns

In [ ]:
dry "SELECT pickup_datetime, passenger_count, fare_amount FROM \`$TAXI\`"

In [ ]:
dry "SELECT AVG(fare_amount) FROM \`$TAXI\`"

## Step 4 · `LIMIT 10`

In [ ]:
dry "SELECT * FROM \`$TAXI\` LIMIT 10"

## Step 5 · `COUNT(*)`

In [ ]:
dry "SELECT COUNT(*) FROM \`$TAXI\`"

In [ ]:
q "SELECT COUNT(*) AS trips FROM \`$TAXI\`"

## Step 6 · Two filters, identical rows

In [ ]:
dry "SELECT pickup_datetime, passenger_count, fare_amount FROM \`$WILD\`
     WHERE _TABLE_SUFFIX = '2022'"

In [ ]:
dry "SELECT pickup_datetime, passenger_count, fare_amount FROM \`$WILD\`
     WHERE EXTRACT(YEAR FROM pickup_datetime) = 2022"

## Step 7 · Partition and cluster, built live

In [ ]:
q "CREATE TABLE \`$DS.trips_2022\` AS
   SELECT vendor_id, pickup_datetime, dropoff_datetime, passenger_count, trip_distance,
          payment_type, fare_amount, tip_amount, total_amount,
          pickup_location_id, dropoff_location_id
   FROM \`$TAXI\` WHERE pickup_datetime >= '2022-01-01' AND pickup_datetime < '2023-01-01'"

In [ ]:
q "CREATE TABLE \`$DS.trips_2022_part\`
   PARTITION BY DATE(pickup_datetime) CLUSTER BY pickup_location_id AS
   SELECT vendor_id, pickup_datetime, dropoff_datetime, passenger_count, trip_distance,
          payment_type, fare_amount, tip_amount, total_amount,
          pickup_location_id, dropoff_location_id
   FROM \`$TAXI\` WHERE pickup_datetime >= '2022-01-01' AND pickup_datetime < '2023-01-01'"

In [ ]:
for t in trips_2022 trips_2022_part; do
  bq --project_id="$PROJECT" show --format=json "$PROJECT:$DATASET.$t" \
    | jq -cM '{table: .tableReference.tableId, numRows, numBytes,
             partitioning: .timePartitioning.type, clustering: .clustering.fields}'
done

In [ ]:
dry "SELECT pickup_location_id, COUNT(*) AS trips, ROUND(SUM(total_amount), 2) AS revenue
     FROM \`$DS.trips_2022\`
     WHERE pickup_datetime >= '2022-07-04' AND pickup_datetime < '2022-07-11'
       AND pickup_location_id = '132'
     GROUP BY 1"

In [ ]:
dry "SELECT pickup_location_id, COUNT(*) AS trips, ROUND(SUM(total_amount), 2) AS revenue
     FROM \`$DS.trips_2022_part\`
     WHERE pickup_datetime >= '2022-07-04' AND pickup_datetime < '2022-07-11'
       AND pickup_location_id = '132'
     GROUP BY 1"

In [ ]:
q "SELECT pickup_location_id, COUNT(*) AS trips, ROUND(SUM(total_amount), 2) AS revenue
   FROM \`$DS.trips_2022\`
   WHERE pickup_datetime >= '2022-07-04' AND pickup_datetime < '2022-07-11'
       AND pickup_location_id = '132'
     GROUP BY 1"

In [ ]:
q "SELECT pickup_location_id, COUNT(*) AS trips, ROUND(SUM(total_amount), 2) AS revenue
   FROM \`$DS.trips_2022_part\`
   WHERE pickup_datetime >= '2022-07-04' AND pickup_datetime < '2022-07-11'
       AND pickup_location_id = '132'
     GROUP BY 1"

In [ ]:
q "ALTER TABLE \`$DS.trips_2022_part\` SET OPTIONS (require_partition_filter = TRUE)"

In [ ]:
q "SELECT COUNT(*) AS trips FROM \`$DS.trips_2022_part\` WHERE pickup_location_id = '132'"

## Step 8 · The execution plan

In [ ]:
q "SELECT pickup_location_id, COUNT(*) AS trips FROM \`$DS.trips_2022\`
   GROUP BY 1 ORDER BY trips DESC LIMIT 5"

In [ ]:
plan

## Step 9 · An external table over Parquet

In [ ]:
gcloud storage ls -l "gs://$BUCKET/trips/2022-01/"

In [ ]:
q "CREATE EXTERNAL TABLE \`$DS.trips_ext\`
   OPTIONS (format = 'PARQUET', uris = ['gs://$BUCKET/trips/2022-01/*.parquet'])"

In [ ]:
for t in trips_ext trips_2022; do
  bq --project_id="$PROJECT" show --format=json "$PROJECT:$DATASET.$t" \
    | jq -cM '{table: .tableReference.tableId, type, numRows, numBytes}'
done

In [ ]:
dry "SELECT payment_type, COUNT(*) AS trips, ROUND(AVG(tip_amount), 2) AS avg_tip
     FROM \`$DS.trips_ext\` GROUP BY 1 ORDER BY trips DESC"

In [ ]:
q "SELECT payment_type, COUNT(*) AS trips, ROUND(AVG(tip_amount), 2) AS avg_tip
     FROM \`$DS.trips_ext\` GROUP BY 1 ORDER BY trips DESC"

## Step 10 · BigLake, delegation, and a policy tag

In [ ]:
q "CREATE EXTERNAL TABLE \`$DS.trips_lake\`
   WITH CONNECTION \`$CONNECTION\`
   OPTIONS (format = 'PARQUET', uris = ['gs://$BUCKET/trips/2022-01/*.parquet'])"

In [ ]:
as_analyst q "SELECT COUNT(*) AS trips FROM \`$DS.trips_ext\`"

In [ ]:
as_analyst q "SELECT COUNT(*) AS trips FROM \`$DS.trips_lake\`"

In [ ]:
bq --project_id="$PROJECT" show --schema --format=json "$PROJECT:$DATASET.trips_lake" \
  | jq --arg tag "$POLICY_TAG" \
       'map(if .name == "pickup_location_id" or .name == "dropoff_location_id"
            then . + {policyTags: {names: [$tag]}} else . end)' \
  > "$WORK/schema-tagged.json"
bq --project_id="$PROJECT" update --schema "$WORK/schema-tagged.json" "$PROJECT:$DATASET.trips_lake"

In [ ]:
as_analyst q "SELECT * FROM \`$DS.trips_lake\` LIMIT 3"

In [ ]:
as_analyst q "SELECT * EXCEPT (pickup_location_id, dropoff_location_id)
              FROM \`$DS.trips_lake\` ORDER BY pickup_datetime LIMIT 3"

In [ ]:
q "SELECT pickup_location_id, COUNT(*) AS trips FROM \`$DS.trips_lake\`
   GROUP BY 1 ORDER BY trips DESC LIMIT 3"

## Step 11 · Row access policies

In [ ]:
q "CREATE ROW ACCESS POLICY vendor_2 ON \`$DS.trips_lake\`
   GRANT TO ('serviceAccount:$ANALYST') FILTER USING (vendor_id = '2')"

In [ ]:
q "SELECT COUNT(*) AS trips, COUNT(DISTINCT vendor_id) AS vendors,
          STRING_AGG(DISTINCT vendor_id ORDER BY vendor_id) AS vendor_ids
   FROM \`$DS.trips_lake\`"

In [ ]:
q "CREATE ROW ACCESS POLICY all_rows ON \`$DS.trips_lake\`
   GRANT TO ('user:$ME') FILTER USING (TRUE)"

In [ ]:
q "SELECT COUNT(*) AS trips, COUNT(DISTINCT vendor_id) AS vendors,
          STRING_AGG(DISTINCT vendor_id ORDER BY vendor_id) AS vendor_ids
   FROM \`$DS.trips_lake\`"

In [ ]:
as_analyst q "SELECT COUNT(*) AS trips, COUNT(DISTINCT vendor_id) AS vendors,
          STRING_AGG(DISTINCT vendor_id ORDER BY vendor_id) AS vendor_ids
   FROM \`$DS.trips_lake\`"

In [ ]:
q "CREATE ROW ACCESS POLICY vendor_2 ON \`$DS.trips_ext\`
   GRANT TO ('serviceAccount:$ANALYST') FILTER USING (vendor_id = '2')"

In [ ]:
bq --project_id="$PROJECT" ls --row_access_policies "$PROJECT:$DATASET.trips_lake"

## Step 12 · Teardown

In [ ]:
bq --project_id="${PROJECT:?}" rm -r -f -d "${DATASET:?}"
bq --project_id="${PROJECT:?}" --location=US rm -f --connection "${CONNECTION:?}"
curl -s -X DELETE -H "Authorization: Bearer $(gcloud auth print-access-token)" \
     -H "x-goog-user-project: ${PROJECT:?}" "https://datacatalog.googleapis.com/v1/${TAXONOMY:?}"
gcloud storage rm -r "gs://${BUCKET:?}"

In [ ]:
bq --project_id="$PROJECT" ls | grep -c "$DATASET"
bq --project_id="$PROJECT" --location=US ls --connection | grep -c "$CONNECTION_ID"
gcloud data-catalog taxonomies list --location us --project "$PROJECT" --format="value(name)" | grep -c .
gcloud storage ls --project "$PROJECT" | grep -c "$BUCKET"